# Apache Arrow + the Arrow IPC / Feather file format

*weyland notebook library (B81) — format deep dive 02*

Everything here is **self-contained**: we build small sample data in the notebook
with `pyarrow` and `polars`, then explore it. No external services, no network,
no cluster. Safe to run top-to-bottom on any singleuser JupyterHub pod.

## Arrow is a *standard*, not (just) a file format

The single most important idea: **Apache Arrow is a specification for how columnar
data is laid out in memory** — the exact bytes, the buffer layout, the null
bitmaps, how strings and lists and structs are encoded. It is a *language-agnostic
memory format*, not a serialized-to-disk format like CSV or Parquet.

Because every Arrow implementation (C++, Python, Rust, Java, Go, R, DuckDB,
Polars, …) agrees on that byte layout, two libraries in the **same process** can
share the *same buffers* with no serialization and no copying. A Polars
`DataFrame`, a pandas frame backed by Arrow, a DuckDB query result, and a
`pyarrow.Table` can all point at one set of bytes. That is what people mean by
"zero-copy" — it is a property of the *in-memory standard*, which the file format
merely happens to mirror on disk.

## The Arrow IPC format (a.k.a. Feather v2)

Arrow *also* defines an **IPC (Inter-Process Communication) format** — a way to
serialize those same in-memory buffers to a byte stream or a file, essentially
verbatim. Two flavours:

| | **IPC File** (`.arrow` / `.feather`) | **IPC Stream** |
|---|---|---|
| Layout | Magic bytes + record batches + **footer** with a block index | Record batches, no footer |
| Random access | **Yes** — footer lets you seek to any batch; **mmap-able** | No — read start to end |
| Use | On-disk file you re-open, memory-map, random-read | Sockets, pipes, "here's a result set" hand-off |
| API | `feather.write_feather`, `ipc.new_file` | `ipc.new_stream` |

**Feather v2 == the Arrow IPC *file* format.** The name "Feather" is historical
(v1 was a different, now-deprecated layout); today `write_feather(...)` writes a
standard Arrow IPC file. We use the terms interchangeably below.

## How this differs from Parquet

Arrow IPC and Parquet look similar (both columnar, both binary) but are optimized
for opposite ends of the data lifecycle:

- **Arrow IPC / Feather** — mirrors the *in-memory* layout. Little to no
  encoding transformation, **no compression by default**, so writing and reading
  are extremely fast and a reader can **memory-map** the file and use the buffers
  in place with **zero deserialization**. Optimized for **speed and interchange**,
  not for footprint. Ephemeral / hot-path data.
- **Parquet** — a *storage* format. Heavy encodings (dictionary, RLE,
  bit-packing, delta) plus compression (snappy/zstd) shrink files dramatically and
  carry rich column statistics for predicate/column pushdown. Optimized for
  **long-term storage, scan-over-cold-data, and network/disk footprint** — at the
  cost of a decode step on every read.

Rule of thumb: **Parquet is how you *store* a table; Arrow is how you *hold and
move* it while working.** The rest of this notebook demonstrates each claim.

## 0 · Environment

Confirm the versions this notebook was written against. All of these ship in the
weyland singleuser image.

In [1]:
import io, os, time, tempfile, pathlib

import numpy as np
import pyarrow as pa
import pyarrow.feather as feather
import pyarrow.ipc as ipc
import pyarrow.parquet as pq
import polars as pl
import pandas as pd
import duckdb

print("pyarrow", pa.__version__)
print("polars ", pl.__version__)
print("duckdb ", duckdb.__version__)
print("pandas ", pd.__version__)
print("numpy  ", np.__version__)

# One scratch dir for every file this notebook writes; cleaned up implicitly
# when the pod dies. Nothing leaves the container.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="arrow_ipc_"))
print("scratch dir:", WORK)

pyarrow 25.0.0
polars  1.44.1
duckdb  1.5.5
pandas  2.3.3
numpy   2.4.6
scratch dir: /tmp/arrow_ipc_8xkuk9f5


## 1 · Build an Arrow Table — schema, chunked arrays, buffers

We synthesize a small "sensor readings" table in-process. No files yet — this is
pure in-memory Arrow. We build it from Arrow arrays directly so we can inspect the
structure the standard defines.

In [2]:
rng = np.random.default_rng(42)
N = 12  # tiny on purpose so we can print the whole thing

sensor_id = pa.array([f"sensor-{i % 3:02d}" for i in range(N)], type=pa.string())
reading   = pa.array(rng.normal(20.0, 1.5, N).round(3), type=pa.float64())
ok        = pa.array(rng.random(N) > 0.2, type=pa.bool_())
# A column WITH nulls, to see the validity bitmap show up in the buffers.
battery   = pa.array([90, 88, None, 71, 60, None, 55, 40, 33, None, 20, 10],
                     type=pa.int16())

table = pa.table(
    {"sensor_id": sensor_id, "reading": reading, "ok": ok, "battery": battery}
)
table

pyarrow.Table
sensor_id: string
reading: double
ok: bool
battery: int16
----
sensor_id: [["sensor-00","sensor-01","sensor-02","sensor-00","sensor-01",...,"sensor-01","sensor-02","sensor-00","sensor-01","sensor-02"]]
reading: [[20.457,18.44,21.126,21.411,17.073,...,19.526,19.975,18.72,21.319,21.167]]
ok: [[true,true,true,true,true,...,true,true,true,true,true]]
battery: [[90,88,null,71,60,...,40,33,null,20,10]]

### Schema

The **schema** is the typed description of every column — Arrow is strongly typed
(unlike raw CSV). Types like `int16`, `float64`, `bool`, `string` are part of the
standard, so `int16` means the same 2-byte layout in every Arrow implementation.

In [3]:
print(table.schema)
print()
print("rows:", table.num_rows, " columns:", table.num_columns)
print("estimated in-memory bytes:", table.nbytes)

sensor_id: string
reading: double
ok: bool
battery: int16

rows: 12  columns: 4
estimated in-memory bytes: 280


### Chunked arrays

An Arrow `Table` column is not a single contiguous array — it is a
**`ChunkedArray`**: a logical column made of one or more physical `Array` chunks.
This is what lets Arrow append/concatenate batches without recopying everything
into one buffer. Our table was built in one shot, so each column currently has a
single chunk — let's create a genuinely multi-chunk column by concatenating two
tables, then look at it.

In [4]:
two = pa.concat_tables([table, table])   # 24 rows, but stored as 2 chunks
col = two.column("reading")
print("type          :", type(col).__name__)
print("logical length:", len(col))
print("num_chunks    :", col.num_chunks)
for i, chunk in enumerate(col.chunks):
    print(f"  chunk[{i}] is a {type(chunk).__name__} of length {len(chunk)}")

type          : ChunkedArray
logical length: 24
num_chunks    : 2
  chunk[0] is a DoubleArray of length 12
  chunk[1] is a DoubleArray of length 12


### Buffers — the actual bytes

Down at the physical level, an Arrow array is a small set of **buffers**. For a
primitive column like `battery` (`int16` with nulls) there are two:

1. a **validity bitmap** — one bit per row, 1 = valid / 0 = null,
2. a **data buffer** — the packed `int16` values.

`None` (null) buffers in the list below are slots the type doesn't use (e.g. a
non-nullable column has no validity buffer). Seeing the raw buffers is the payoff
of Arrow being a *memory spec*: this is exactly what gets memory-mapped from a
Feather file later, byte-for-byte.

In [5]:
battery_arr = table.column("battery").combine_chunks()  # single Array
bufs = battery_arr.buffers()
for i, b in enumerate(bufs):
    if b is None:
        print(f"buffer[{i}]: None (unused by this type)")
    else:
        print(f"buffer[{i}]: {b.size:>3} bytes  ->  {bytes(b)!r}")

print()
print("null_count:", battery_arr.null_count, "of", len(battery_arr), "rows")

buffer[0]:   2 bytes  ->  b'\xdb\r'
buffer[1]:  24 bytes  ->  b'Z\x00X\x00\x00\x00G\x00<\x00\x00\x007\x00(\x00!\x00\x00\x00\x14\x00\n\x00'

null_count: 3 of 12 rows


## 2 · Zero-copy interop: pyarrow ↔ polars ↔ pandas ↔ duckdb

The headline benefit. Because all four libraries speak the Arrow memory format,
handing data between them in one process can avoid copying the value buffers.

### What "zero-copy" actually means here

The receiving library wraps the **same underlying Arrow buffers** instead of
allocating new memory and memcpy-ing the values across. It is zero-*copy* of the
bulk data; a tiny bit of metadata (schema, chunk pointers) is always duplicated.
It works when the two sides agree on the exact layout and neither needs to mutate
in place.

It **stops being zero-copy** when a representation conversion is unavoidable:
- a type that has no identical layout on the other side (e.g. Arrow strings →
  NumPy `object` in classic pandas; Arrow's null bitmap → pandas needs NaN/NaT or
  a masked type),
- **nulls in a numeric column** going to a NumPy-backed pandas column (NumPy ints
  can't hold NA, so a copy + upcast to float is forced),
- a chunked column being materialized into one contiguous array.

### polars: `pl.from_arrow`

In [6]:
pldf = pl.from_arrow(table)   # wraps Arrow buffers; Polars is Arrow-native
print(type(pldf).__name__)
pldf

DataFrame


sensor_id,reading,ok,battery
str,f64,bool,i16
"""sensor-00""",20.457,true,90
"""sensor-01""",18.44,true,88
"""sensor-02""",21.126,true,null
"""sensor-00""",21.411,true,71
"""sensor-01""",17.073,true,60
…,…,…,…
"""sensor-01""",19.526,true,40
"""sensor-02""",19.975,true,33
"""sensor-00""",18.72,true,null


Polars uses the Arrow memory model internally, so `pl.from_arrow` is a
zero-copy wrap of the value buffers (the `reading`/`ok`/`battery` bytes are not
re-encoded). Going back out is symmetric:

In [7]:
back = pldf.to_arrow()
print("round-tripped back to:", type(back).__name__)
print("schema preserved:", back.schema.equals(table.schema))

round-tripped back to: Table
schema preserved: False


### duckdb: query an Arrow table in place

DuckDB can scan a `pyarrow.Table` **directly** — the table name in the SQL
resolves to the Python variable (a *replacement scan*), and DuckDB reads the Arrow
buffers without an import/copy step. This is the pattern you'll use constantly:
keep data in Arrow, push the compute into DuckDB's vectorized engine.

> Our variable is literally named `table`, which is a **reserved word** in SQL, so
> we double-quote it (`FROM "table"`) to use it as an identifier. Name your Arrow
> variable something un-reserved (`readings`, `df`, …) and you can drop the
> quotes entirely.

In [8]:
con = duckdb.connect()  # in-memory database

# `table` (our pyarrow.Table) is referenced straight from SQL. No load step.
res = con.sql('''
    SELECT sensor_id,
           COUNT(*)        AS n,
           ROUND(AVG(reading), 3) AS avg_reading,
           SUM(ok::INT)    AS ok_count
    FROM "table"
    GROUP BY sensor_id
    ORDER BY sensor_id
''')
res.show()

┌───────────┬───────┬─────────────┬──────────┐
│ sensor_id │   n   │ avg_reading │ ok_count │
│  varchar  │ int64 │   double    │  int128  │
├───────────┼───────┼─────────────┼──────────┤
│ sensor-00 │     4 │      20.195 │        4 │
│ sensor-01 │     4 │       19.09 │        4 │
│ sensor-02 │     4 │      20.079 │        3 │
└───────────┴───────┴─────────────┴──────────┘



And the result comes back out as Arrow just as easily (`.arrow()`), or as
Polars (`.pl()`) / pandas (`.df()`) — DuckDB is Arrow-native on both the input and
output side.

In [9]:
agg_arrow = con.sql('SELECT sensor_id, AVG(reading) AS avg_reading '
                    'FROM "table" GROUP BY sensor_id ORDER BY sensor_id').arrow()
print("duckdb -> Arrow:", type(agg_arrow).__name__)
print(agg_arrow.schema)

duckdb -> Arrow: RecordBatchReader
sensor_id: string
avg_reading: double


### pandas: `to_pandas(zero_copy_only=...)`

pandas is the interesting case because classic pandas is **NumPy-backed**, and
NumPy's layout does *not* match Arrow's for several types. `to_pandas(...)`
accepts a `zero_copy_only` flag that turns the silent copy into a **loud error**,
which is a great way to *learn which columns can and cannot* be shared for free.

First, force the strict mode on the whole table — it should **fail**, because
`sensor_id` (string) and `battery` (int16 **with nulls**) cannot be represented
zero-copy in NumPy-backed pandas:

In [10]:
try:
    table.to_pandas(zero_copy_only=True)
except Exception as e:
    print(type(e).__name__, "->", e)

ArrowInvalid -> Cannot do zero copy conversion into multi-column DataFrame block


Now isolate a column that *can* go zero-copy: a numeric column with **no
nulls** and no chunking maps straight onto a NumPy array over the same buffer.
`reading` (float64, no nulls) qualifies:

In [11]:
reading_col = table.column("reading").combine_chunks()  # single chunk
s = reading_col.to_pandas(zero_copy_only=True)             # succeeds
print("zero-copy Series dtype:", s.dtype)
print(s.head().to_list())

zero-copy Series dtype: float64
[20.457, 18.44, 21.126, 21.411, 17.073]


Contrast: ask the same of `battery` (has nulls) in strict mode and it
refuses — NumPy int can't hold NA, so pandas would have to copy and upcast to
`float64`. Drop `zero_copy_only` and it succeeds *by making that copy*.

In [12]:
bat = table.column("battery").combine_chunks()
try:
    bat.to_pandas(zero_copy_only=True)
except Exception as e:
    print("strict  :", type(e).__name__, "->", e)

lenient = bat.to_pandas()  # allowed to copy -> upcasts to float64 with NaN
print("lenient :", lenient.dtype, lenient.to_list())

strict  : ArrowInvalid -> Needed to copy 1 chunks with 3 nulls, but zero_copy_only was True
lenient : float64 [90.0, 88.0, nan, 71.0, 60.0, nan, 55.0, 40.0, 33.0, nan, 20.0, 10.0]


### The modern escape hatch: Arrow-backed pandas

pandas can also store columns in **Arrow arrays directly** (`dtype_backend=
"pyarrow"`), sidestepping the NumPy mismatch entirely — strings and
nullable ints then keep their Arrow layout and preserve nulls as real `NA`.

In [13]:
arrow_backed = table.to_pandas(types_mapper=pd.ArrowDtype)
print(arrow_backed.dtypes)
print()
print("battery keeps nulls as <NA>, not NaN:")
print(arrow_backed["battery"].to_list())

sensor_id    string[pyarrow]
reading      double[pyarrow]
ok             bool[pyarrow]
battery       int16[pyarrow]
dtype: object

battery keeps nulls as <NA>, not NaN:
[90, 88, <NA>, 71, 60, <NA>, 55, 40, 33, <NA>, 20, 10]


## 3 · IPC file vs stream, and memory-mapping

Now we serialize. We'll write the **same table** three ways and read each back:

1. `feather.write_feather` — the friendly one-liner (writes an Arrow IPC *file*),
2. `ipc.new_file` — the explicit IPC **file** writer (footer + block index → random
   access + mmap),
3. `ipc.new_stream` — the IPC **stream** writer (no footer → sequential only).

To make the difference between "hold the file" and "mmap the file" visible, we
scale up to a bigger table first.

In [14]:
big = pa.table({
    "id":     pa.array(np.arange(500_000, dtype=np.int64)),
    "value":  pa.array(rng.normal(0, 1, 500_000)),
    "label":  pa.array(rng.integers(0, 5, 500_000).astype("U1")),
})
print("big table:", big.num_rows, "rows,", f"{big.nbytes/1e6:.1f} MB in memory")

big table: 500000 rows, 10.5 MB in memory


### 3a · `feather.write_feather` — the easy path

In [15]:
feather_path = WORK / "big.feather"
feather.write_feather(big, feather_path)      # uncompressed Arrow IPC file
print("wrote", feather_path.name, f"({feather_path.stat().st_size/1e6:.1f} MB on disk)")

rt = feather.read_table(feather_path)         # read fully into memory
print("read back:", rt.num_rows, "rows; schema matches:", rt.schema.equals(big.schema))

wrote big.feather (8.4 MB on disk)
read back: 500000 rows; schema matches: True


/tmp/ipykernel_3017900/4009050137.py:2: FutureWarning: pyarrow.feather.write_feather is deprecated as of 24.0.0. Use pyarrow.ipc.new_file() / RecordBatchFileWriter instead. Feather V2 is the Arrow IPC file format.
  feather.write_feather(big, feather_path)      # uncompressed Arrow IPC file
/tmp/ipykernel_3017900/4009050137.py:5: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  rt = feather.read_table(feather_path)         # read fully into memory


### 3b · `ipc.new_file` — explicit IPC **file** (record batches + footer)

The file writer lets us control record-batch boundaries and produces a footer
with a **block index**. That index is what enables `get_record_batch(i)` random
access and memory-mapping.

In [16]:
ipc_file_path = WORK / "explicit_file.arrow"
batches = big.to_batches(max_chunksize=125_000)   # 4 record batches

with pa.OSFile(str(ipc_file_path), "wb") as sink:
    with ipc.new_file(sink, big.schema) as writer:
        for b in batches:
            writer.write_batch(b)
print("wrote", ipc_file_path.name, "with", len(batches), "record batches")

# Random access: open and seek to a specific batch WITHOUT reading the others.
with pa.memory_map(str(ipc_file_path), "r") as source:
    reader = ipc.open_file(source)               # 'open_file' == reading IPC FILE
    print("reader sees num_record_batches:", reader.num_record_batches)
    third = reader.get_record_batch(2)           # jump straight to batch #2
    print("batch[2]:", third.num_rows, "rows, first id =", third.column("id")[0].as_py())

wrote explicit_file.arrow with 4 record batches
reader sees num_record_batches: 4
batch[2]: 125000 rows, first id = 250000


### 3c · `ipc.new_stream` — IPC **stream** (no footer)

The stream format writes the same record batches but **no footer/index**. You can
only read it front-to-back with `open_stream` — there is no `get_record_batch(i)`
and it cannot be memory-mapped for random access. This is the format you'd push
over a socket or pipe.

In [17]:
stream_path = WORK / "data.arrows"
with pa.OSFile(str(stream_path), "wb") as sink:
    with ipc.new_stream(sink, big.schema) as writer:
        for b in batches:
            writer.write_batch(b)
print("wrote", stream_path.name)

# Stream readers are sequential: iterate batches, no random seek.
with pa.OSFile(str(stream_path), "rb") as source:
    reader = ipc.open_stream(source)
    seen = [b.num_rows for b in reader]           # consume front-to-back
print("stream delivered", len(seen), "batches, total rows:", sum(seen))
print("open_file on a STREAM file would raise -- the footer/magic isn't there.")

wrote data.arrows
stream delivered 4 batches, total rows: 500000
open_file on a STREAM file would raise -- the footer/magic isn't there.


### 3d · Memory-mapping a Feather file — the big one

Reading normally **copies file bytes into process RAM**. Memory-mapping instead
maps the file's pages into the address space; because Feather's on-disk layout
*is* the Arrow memory layout, the Arrow arrays can point **directly at the mapped
pages**. Benefits:

- **No up-front deserialization / decode** — unlike Parquet, there's nothing to
  decompress or un-encode; the bytes are already Arrow.
- **Lazy, paged loading** — the OS pages in only the parts you actually touch.
- **Shared across processes** — N readers mmap one file and share the same
  physical pages, instead of each holding a private copy. Great for a read-only
  reference table loaded by many workers.

`feather.read_table(..., memory_map=True)` (or `pa.memory_map(...)`) opts in.

In [18]:
# Full read: bytes copied into RAM.
t0 = time.perf_counter()
mem_tbl = feather.read_table(feather_path, memory_map=False)
t_mem = time.perf_counter() - t0

# Memory-mapped: arrays reference the mapped file pages, no bulk copy.
t0 = time.perf_counter()
mmap_tbl = feather.read_table(feather_path, memory_map=True)
t_mmap = time.perf_counter() - t0

print(f"read (copy into RAM) : {t_mem*1e3:8.2f} ms")
print(f"read (memory-mapped) : {t_mmap*1e3:8.2f} ms")
print("same data:", mmap_tbl.num_rows == mem_tbl.num_rows == big.num_rows)
print()
print("Timings are tiny at this size; the real mmap win is (a) not paying a")
print("decode step and (b) many processes sharing one file's pages instead of")
print("each allocating a private", f"{big.nbytes/1e6:.0f} MB copy.")

read (copy into RAM) :     4.09 ms
read (memory-mapped) :     3.59 ms
same data: True

Timings are tiny at this size; the real mmap win is (a) not paying a
decode step and (b) many processes sharing one file's pages instead of
each allocating a private 10 MB copy.


/tmp/ipykernel_3017900/3790518666.py:3: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  mem_tbl = feather.read_table(feather_path, memory_map=False)
/tmp/ipykernel_3017900/3790518666.py:8: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  mmap_tbl = feather.read_table(feather_path, memory_map=True)


We can also mmap by hand to prove the file bytes *are* the Arrow bytes —
open the mapping, read the IPC file, and the columns work with no separate load
step:

In [19]:
import pyarrow.compute as pc

with pa.memory_map(str(feather_path), "r") as mm:
    tbl = ipc.open_file(mm).read_all()
    # Compute straight off the mapped pages -- no separate load step.
    mean_value = pc.mean(tbl.column("value")).as_py()
    print("mean(value) over mmapped file:", round(mean_value, 4))

mean(value) over mmapped file: -0.0005


## 4 · Arrow IPC vs Parquet — size and speed on the same data

The trade-off made concrete. We write `big` to:

- **Feather uncompressed** — pure speed, largest file,
- **Feather + LZ4 / ZSTD** — IPC *does* support optional compression per buffer,
- **Parquet (snappy)** and **Parquet (zstd)** — storage-optimized.

…and time a full write + read for each, plus the resulting file size.

In [20]:
import pyarrow.compute as pc

def bench(label, write_fn, read_fn, path):
    t0 = time.perf_counter(); write_fn(path); t_w = time.perf_counter() - t0
    t0 = time.perf_counter(); tbl = read_fn(path); t_r = time.perf_counter() - t0
    size = path.stat().st_size
    assert tbl.num_rows == big.num_rows
    return {"format": label,
            "write_ms": round(t_w * 1e3, 1),
            "read_ms":  round(t_r * 1e3, 1),
            "MB":       round(size / 1e6, 2)}

rows = []
rows.append(bench(
    "feather (none)",
    lambda p: feather.write_feather(big, p, compression="uncompressed"),
    lambda p: feather.read_table(p),
    WORK / "b_none.feather"))
rows.append(bench(
    "feather (lz4)",
    lambda p: feather.write_feather(big, p, compression="lz4"),
    lambda p: feather.read_table(p),
    WORK / "b_lz4.feather"))
rows.append(bench(
    "feather (zstd)",
    lambda p: feather.write_feather(big, p, compression="zstd"),
    lambda p: feather.read_table(p),
    WORK / "b_zstd.feather"))
rows.append(bench(
    "parquet (snappy)",
    lambda p: pq.write_table(big, p, compression="snappy"),
    lambda p: pq.read_table(p),
    WORK / "b_snappy.parquet"))
rows.append(bench(
    "parquet (zstd)",
    lambda p: pq.write_table(big, p, compression="zstd"),
    lambda p: pq.read_table(p),
    WORK / "b_zstd.parquet"))

pl.DataFrame(rows)

/tmp/ipykernel_3017900/2585256840.py:16: FutureWarning: pyarrow.feather.write_feather is deprecated as of 24.0.0. Use pyarrow.ipc.new_file() / RecordBatchFileWriter instead. Feather V2 is the Arrow IPC file format.
  lambda p: feather.write_feather(big, p, compression="uncompressed"),
/tmp/ipykernel_3017900/2585256840.py:17: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  lambda p: feather.read_table(p),
/tmp/ipykernel_3017900/2585256840.py:21: FutureWarning: pyarrow.feather.write_feather is deprecated as of 24.0.0. Use pyarrow.ipc.new_file() / RecordBatchFileWriter instead. Feather V2 is the Arrow IPC file format.
  lambda p: feather.write_feather(big, p, compression="lz4"),
/tmp/ipykernel_3017900/2585256840.py:22: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the

format,write_ms,read_ms,MB
str,f64,f64,f64
"""feather (none)""",2.4,1.7,10.5
"""feather (lz4)""",5.3,4.1,8.36
"""feather (zstd)""",8.2,4.2,5.73
"""parquet (snappy)""",30.6,6.1,6.74
"""parquet (zstd)""",36.7,5.8,5.04


Read the table above with the trade-off in mind (exact numbers vary
per run and per host):

- **Uncompressed Feather** is typically the **fastest write and read** and the
  **largest file** — it does almost no work, just moves buffers.
- **Feather + zstd** shrinks the file toward Parquet territory while keeping IPC's
  simple structure; a middle ground when you want a smaller hot-path artifact.
- **Parquet** is usually the **smallest on disk** (columnar encodings + column
  stats do more than block compression alone) but pays a **decode cost** on read
  and cannot be memory-mapped into usable arrays the way Feather can.

Same data, opposite optimizations: **fewest bytes** (Parquet) vs **fewest CPU
cycles between disk and a usable column** (Feather).

## 5 · When to reach for which format

A cheat-sheet for the weyland lab.

### Use Arrow IPC / Feather when…
- **Fast interchange between tools/processes** on one box — Polars → DuckDB →
  pandas hand-offs, or spilling an intermediate result to disk and picking it back
  up. It's the "hold and move" format.
- You want to **memory-map** a read-only reference table shared by many workers,
  or page a large table in lazily without a decode step.
- The data is **ephemeral / hot** — a cache, a scratch checkpoint, a result set —
  where read/write speed matters more than footprint and you'll delete it soon.
- You need an **exact, loss-free round-trip** of Arrow types with no re-encoding.

### Use Parquet when…
- **Long-term storage** in the lakehouse (lakeFS / Iceberg / object storage).
  Smaller files, column statistics, predicate & column pushdown on cold scans.
- Data is **written once, read many times, over the network** — footprint and
  scan efficiency dominate.
- You need broad ecosystem interop for *analytics at rest* (Trino, Spark, DuckDB,
  every warehouse reads Parquet).

### Use Avro when…
- **Row-oriented streaming / messaging** (Kafka/Redpanda events), where records
  arrive one at a time and **schema evolution** with a registry is the priority —
  not columnar scan performance.

### Use Lance when…
- **ML / vector workloads** — random access to rows by index, fast vector search,
  versioned datasets, and columnar reads tuned for feature retrieval and
  embeddings. It targets the access pattern training/serving needs, which neither
  Parquet nor Arrow IPC is built for.

### One-line mental model
> **Avro** moves rows. **Parquet** stores columns cheaply. **Lance** serves
> ML/vectors. **Arrow** is the in-memory standard that lets all of them — and
> Polars, DuckDB, pandas — share the *same bytes* while you work, with Feather as
> its fast, mmap-able on-disk twin.